In [1]:
# ============================================================
# CELL 1: Install dependencies and check GPU
# ============================================================
!pip install -q scikit-learn xgboost tensorflow pandas numpy scipy

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpu = tf.config.list_physical_devices('GPU')
if gpu:
    print(f'GPU available: {gpu[0]}')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU')

TensorFlow: 2.19.0


In [2]:
# ============================================================
# CELL 2: Upload your dataset
# ============================================================
# Upload Final_ITD.csv (the preprocessed file your thesis models used)
from google.colab import files
uploaded = files.upload()
DATA_FILENAME = list(uploaded.keys())[0]
print(f'\nUploaded: {DATA_FILENAME}')
print(f'Size: {len(uploaded[DATA_FILENAME]):,} bytes')

Saving Final_ITD.csv to Final_ITD.csv

Uploaded: Final_ITD.csv
Size: 14,146,431 bytes


CELL 3: Import all **Libraries** **bold text**

In [3]:
# ============================================================
# CELL 3: Import all libraries
# ============================================================
import os

# Force determinism (MUST be before importing TensorFlow)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['PYTHONHASHSEED'] = '42'

import warnings
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.inspection import permutation_importance
from scipy.stats import wilcoxon, spearmanr
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

os.makedirs('results', exist_ok=True)

print('All imports successful.')
print(f'TensorFlow: {tf.__version__}')
print('Deterministic ops enabled.')

All imports successful.
TensorFlow: 2.19.0
Deterministic ops enabled.


**CELL 4: Configure dataset columns**

In [4]:
# ============================================================
# CELL 4: Configure dataset columns
# ============================================================
TARGET_COL = 'is_emp_malicious'
DROP_COLS = ['employee_id', 'date', 'trip_day_number', 'country_name']
CATEGORICAL_COLS = []

print('Configuration set.')
print(f'  Target column: {TARGET_COL}')
print(f'  Columns to drop: {DROP_COLS}')

Configuration set.
  Target column: is_emp_malicious
  Columns to drop: ['employee_id', 'date', 'trip_day_number', 'country_name']


In [5]:
# ============================================================
# CELL 5: Load and inspect dataset
# ============================================================
import pandas as pd
import os

# Check if the filename variable exists from CELL 2
try:
    df_raw = pd.read_csv(DATA_FILENAME)

    print(f'Dataset shape: {df_raw.shape}')
    print(f'\nColumns ({len(df_raw.columns)}):')
    for i, col in enumerate(df_raw.columns):
        print(f'  {i+1:2d}. {col} ({df_raw[col].dtype})')

    print(f'\nFirst 3 rows:')
    display(df_raw.head(3))
except NameError:
    print("ERROR: DATA_FILENAME is not defined. Please run CELL 2 first and upload your dataset.")
except FileNotFoundError:
    print(f"ERROR: The file '{DATA_FILENAME}' was not found. Please re-upload it in CELL 2.")

Dataset shape: (29880, 38)

Columns (38):
   1. employee_seniority_years (float64)
   2. is_contractor (float64)
   3. employee_classification (float64)
   4. has_foreign_citizenship (float64)
   5. has_criminal_record (float64)
   6. has_medical_history (float64)
   7. risk_travel_indicator (float64)
   8. num_print_commands (float64)
   9. total_printed_pages (float64)
  10. num_print_commands_off_hours (float64)
  11. num_printed_pages_off_hours (float64)
  12. num_color_prints (float64)
  13. num_bw_prints (float64)
  14. ratio_color_prints (float64)
  15. printed_from_other (float64)
  16. print_campuses (float64)
  17. num_burn_requests (float64)
  18. max_request_classification (float64)
  19. avg_request_classification (float64)
  20. num_burn_requests_off_hours (float64)
  21. total_burn_volume_mb (float64)
  22. total_files_burned (float64)
  23. burned_from_other (float64)
  24. burn_campuses (float64)
  25. is_abroad (float64)
  26. is_hostile_country_trip (float64)
  27. h

,employee_seniority_years,is_contractor,employee_classification,has_foreign_citizenship,has_criminal_record,has_medical_history,risk_travel_indicator,num_print_commands,total_printed_pages,num_print_commands_off_hours,...,num_entries,num_exits,total_presence_minutes,entered_during_night_hours,num_unique_campus,early_entry_flag,entry_during_weekend,first_entry_minutes,last_exit_minutes,is_emp_malicious
0,0.547490,-0.375706,-0.048271,-0.477401,-0.198891,-0.436447,-0.008182,-0.592447,-0.347391,-0.172272,...,-0.999648,-0.999648,-1.187573,-0.039692,-1.069308,-0.069831,-0.29653,-0.114514,-0.181596,1
1,-0.653307,2.661657,-1.388731,-0.477401,5.027891,-0.436447,-0.008182,-0.592447,-0.347391,-0.172272,...,1.392878,1.392878,0.426682,-0.039692,0.481181,-0.069831,-0.29653,-0.159694,-0.993543,0
2,-1.320417,-0.375706,-0.048271,2.094676,-0.198891,-0.436447,-0.008182,1.381759,2.387795,-0.172272,...,1.392878,1.392878,0.564441,-0.039692,0.481181,-0.069831,-0.29653,-0.498547,-0.741181,1


In [6]:
# ============================================================
# CELL 6: Load preprocessed dataset
# Final_ITD.csv is already balanced + StandardScaler normalised
# ============================================================
df = df_raw.copy()
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Data is already preprocessed (StandardScaler + undersampled)')

X_full = df.drop(columns=[TARGET_COL])
y_full = df[TARGET_COL].values
feature_names = X_full.columns.tolist()

print(f'\nFeatures: {X_full.shape[1]}')
print(f'Normal (0): {(y_full == 0).sum():,}')
print(f'Malicious (1): {(y_full == 1).sum():,}')

X = X_full.values
y = y_full

print(f'\nFinal dataset: {X.shape[0]:,} rows, {X.shape[1]} features')
print('Ready for experiments.')

Loaded: 29,880 rows, 38 columns
Data is already preprocessed (StandardScaler + undersampled)

Features: 37
Normal (0): 14,940
Malicious (1): 14,940

Final dataset: 29,880 rows, 37 features
Ready for experiments.


In [7]:
# ============================================================
# CELL 7: Model builders (matching original thesis code)
# ============================================================

def build_rf():
    """RF with GridSearchCV matching original thesis code"""
    rf = RandomForestClassifier(random_state=SEED)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt']
    }
    grid = GridSearchCV(
        estimator=rf, param_grid=param_grid,
        cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
    )
    return grid

def build_xgb():
    """XGBoost with GridSearchCV matching original thesis code"""
    xgb = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=SEED
    )
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
        'gamma': [0, 0.1, 0.2]
    }
    grid = GridSearchCV(
        estimator=xgb, param_grid=param_grid,
        cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
    )
    return grid

def build_mlp_pipeline():
    """sklearn MLP with Pipeline matching original thesis code"""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(max_iter=500, random_state=SEED))
    ])
    param_grid = {
        'mlp__hidden_layer_sizes': [(64,), (128,)],
        'mlp__activation': ['relu'],
        'mlp__solver': ['adam'],
        'mlp__alpha': [0.0001, 0.001],
        'mlp__learning_rate': ['constant']
    }
    grid = GridSearchCV(
        estimator=pipeline, param_grid=param_grid,
        cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
    )
    return grid

def build_ae_mlp_keras(input_dim, encoding_dim=16):
    """AE-MLP matching original thesis (decoder activation=linear)"""
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(32, activation='relu')(input_layer)
    encoded = Dense(encoding_dim, activation='relu')(encoded)
    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(input_dim, activation='linear')(decoded)
    autoencoder = Model(input_layer, decoded)
    encoder = Model(input_layer, encoded)
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    mlp = Sequential([
        Dense(32, activation='relu', input_shape=(encoding_dim,)),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    mlp.compile(optimizer=Adam(learning_rate=0.001),
                loss='binary_crossentropy', metrics=['accuracy'])
    return autoencoder, encoder, mlp

print('Model builders defined (matching original thesis configurations).')

Model builders defined (matching original thesis configurations).


In [8]:
# ============================================================
# CELL 8: Training and evaluation functions
# ============================================================

def evaluate_model(y_true, y_pred, y_prob):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_prob)
    }

def train_eval_rf(X_train, y_train, X_test, y_test):
    """Train RF with GridSearchCV"""
    grid = build_rf()
    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    y_pred = best.predict(X_test)
    y_prob = best.predict_proba(X_test)[:, 1]
    metrics = evaluate_model(y_test, y_pred, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    print(f'Best: {grid.best_params_}')
    return metrics, cm, best

def train_eval_xgb(X_train, y_train, X_test, y_test):
    """Train XGBoost with GridSearchCV"""
    grid = build_xgb()
    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    y_pred = best.predict(X_test)
    y_prob = best.predict_proba(X_test)[:, 1]
    metrics = evaluate_model(y_test, y_pred, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    print(f'Best: {grid.best_params_}')
    return metrics, cm, best

def train_eval_mlp(X_train, y_train, X_test, y_test):
    """Train sklearn MLP Pipeline with GridSearchCV"""
    grid = build_mlp_pipeline()
    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    y_pred = best.predict(X_test)
    y_prob = best.predict_proba(X_test)[:, 1]
    metrics = evaluate_model(y_test, y_pred, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    print(f'Best: {grid.best_params_}')
    return metrics, cm, best

def train_eval_ae_mlp(X_train, y_train, X_test, y_test, input_dim, encoding_dim=16):
    """Train AE-MLP matching original thesis code"""
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    autoencoder, encoder, mlp = build_ae_mlp_keras(input_dim, encoding_dim)
    autoencoder.fit(X_train_s, X_train_s, epochs=50, batch_size=32,
                    validation_split=0.1, verbose=0)
    X_train_enc = encoder.predict(X_train_s, verbose=0)
    X_test_enc = encoder.predict(X_test_s, verbose=0)
    mlp.fit(X_train_enc, y_train, epochs=50, batch_size=32,
            validation_split=0.1, verbose=0)
    y_prob = mlp.predict(X_test_enc, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluate_model(y_test, y_pred, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    return metrics, cm, (autoencoder, encoder, mlp, scaler)

print('Training functions defined.')

Training functions defined.


---
## EXPERIMENT 1: Single Split (Tables 6 and 7)

In [9]:
# ============================================================
# CELL 9: EXPERIMENT 1 - Single 80:20 Split
# ============================================================
print('=' * 70)
print('EXPERIMENT 1: Single Train-Test Split (80:20)')
print('=' * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
input_dim = X_train.shape[1]
print(f'Train: {len(y_train)} | Test: {len(y_test)}')
print(f'Test malicious: {(y_test==1).sum()} | Test normal: {(y_test==0).sum()}')

# Random Forest
print('\nTraining Random Forest (GridSearchCV)...')
rf_metrics, rf_cm, rf_fitted = train_eval_rf(X_train, y_train, X_test, y_test)
print(f'  ROC-AUC = {rf_metrics["roc_auc"]:.4f}')

# XGBoost
print('\nTraining XGBoost (GridSearchCV)...')
xgb_metrics, xgb_cm, xgb_fitted = train_eval_xgb(X_train, y_train, X_test, y_test)
print(f'  ROC-AUC = {xgb_metrics["roc_auc"]:.4f}')

# MLP
print('\nTraining MLP (sklearn Pipeline + GridSearchCV)...')
mlp_metrics, mlp_cm, mlp_fitted = train_eval_mlp(X_train, y_train, X_test, y_test)
print(f'  ROC-AUC = {mlp_metrics["roc_auc"]:.4f}')

# AE-MLP
print('\nTraining AE-MLP (Keras, d=16)...')
ae_metrics, ae_cm, ae_models = train_eval_ae_mlp(
    X_train, y_train, X_test, y_test, input_dim, encoding_dim=16
)
print(f'  ROC-AUC = {ae_metrics["roc_auc"]:.4f}')

print('\nAll models trained.')

EXPERIMENT 1: Single Train-Test Split (80:20)
Train: 23904 | Test: 5976
Test malicious: 2988 | Test normal: 2988

Training Random Forest (GridSearchCV)...
Best: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
  ROC-AUC = 0.9458

Training XGBoost (GridSearchCV)...
Best: {'colsample_bytree': 1.0, 'gamma': 0, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 300, 'subsample': 0.8}
  ROC-AUC = 0.9691

Training MLP (sklearn Pipeline + GridSearchCV)...
Best: {'mlp__activation': 'relu', 'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (128,), 'mlp__learning_rate': 'constant', 'mlp__solver': 'adam'}
  ROC-AUC = 0.9198

Training AE-MLP (Keras, d=16)...
  ROC-AUC = 0.8686

All models trained.


# CELL 10: Print TABLE 6 - Model Test Performance

In [10]:
# ============================================================
# CELL 10: Print TABLE 6 - Model Test Performance
# ============================================================
print('=' * 70)
print('TABLE 6: Model Test Performance (Single 80:20 Split)')
print('=' * 70)

all_metrics = {
    'Random Forest': rf_metrics,
    'XGBoost': xgb_metrics,
    'MLP': mlp_metrics,
    'AE-MLP': ae_metrics
}

print(f'{"Model":<18} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1-Score":>10} {"ROC-AUC":>10}')
print('-' * 68)
for name, m in all_metrics.items():
    print(f'{name:<18} {m["accuracy"]:>10.4f} {m["precision"]:>10.4f} '
          f'{m["recall"]:>10.4f} {m["f1"]:>10.4f} {m["roc_auc"]:>10.4f}')

df_t6 = pd.DataFrame(all_metrics).T
df_t6.index.name = 'Model'
df_t6.to_csv('results/table6_test_performance.csv')
print('\nSaved to results/table6_test_performance.csv')

TABLE 6: Model Test Performance (Single 80:20 Split)
Model                Accuracy  Precision     Recall   F1-Score    ROC-AUC
--------------------------------------------------------------------
Random Forest          0.8638     0.8715     0.8534     0.8624     0.9458
XGBoost                0.8944     0.8888     0.9016     0.8952     0.9691
MLP                    0.8293     0.8610     0.7855     0.8215     0.9198
AE-MLP                 0.7818     0.8680     0.6647     0.7528     0.8686

Saved to results/table6_test_performance.csv


# CELL 11: Print TABLE 7 - Confusion Matrix Decomposition


In [11]:
# ============================================================
# CELL 11: Print TABLE 7 - Confusion Matrix Decomposition
# ============================================================
print('=' * 70)
print(f'TABLE 7: Confusion Matrix Decomposition (Test Set: {len(y_test)} Records)')
print('=' * 70)

all_cm = {
    'XGBoost': xgb_cm,
    'Random Forest': rf_cm,
    'MLP': mlp_cm,
    'AE-MLP': ae_cm
}

print(f'{"Model":<18} {"TP":>8} {"FP":>8} {"TN":>8} {"FN":>8} {"Total":>8}')
print('-' * 52)

cm_data = []
for name, cm in all_cm.items():
    tn, fp, fn, tp = cm.ravel()
    total = tn + fp + fn + tp
    print(f'{name:<18} {tp:>8,} {fp:>8,} {tn:>8,} {fn:>8,} {total:>8,}')
    cm_data.append({'Model': name, 'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn, 'Total': total})

df_t7 = pd.DataFrame(cm_data)
df_t7.to_csv('results/table7_confusion_matrices.csv', index=False)
print('\nSaved to results/table7_confusion_matrices.csv')

TABLE 7: Confusion Matrix Decomposition (Test Set: 5976 Records)
Model                    TP       FP       TN       FN    Total
----------------------------------------------------
XGBoost               2,694      337    2,651      294    5,976
Random Forest         2,550      376    2,612      438    5,976
MLP                   2,347      379    2,609      641    5,976
AE-MLP                1,986      302    2,686    1,002    5,976

Saved to results/table7_confusion_matrices.csv


---
## EXPERIMENT 2: Five-Fold Cross-Validation (Table 9)

In [12]:
# ============================================================
# CELL 12: EXPERIMENT 2 - Five-Fold CV (fixed best params)
# ============================================================
print('=' * 70)
print('EXPERIMENT 2: Five-Fold Stratified Cross-Validation')
print('Using best hyperparameters from Experiment 1')
print('=' * 70)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
input_dim = X.shape[1]

cv_results = {
    'Random Forest': {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []},
    'XGBoost': {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []},
    'MLP': {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []},
    'AE-MLP (d=16)': {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []}
}

# Print locked params
print(f'\nRF params: {rf_fitted.get_params()}')
print(f'XGB params: {xgb_fitted.get_params()}')

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f'\n--- Fold {fold}/5 ---')
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    # RF (fixed params)
    print('  RF...', end=' ')
    rf_fold = clone(rf_fitted)
    rf_fold.fit(X_tr, y_tr)
    y_pred = rf_fold.predict(X_te)
    y_prob = rf_fold.predict_proba(X_te)[:, 1]
    m = evaluate_model(y_te, y_pred, y_prob)
    for k, v in m.items(): cv_results['Random Forest'][k].append(v)
    print(f'AUC={m["roc_auc"]:.4f}', end='  ')

    # XGBoost (fixed params)
    print('XGB...', end=' ')
    xgb_fold = clone(xgb_fitted)
    xgb_fold.fit(X_tr, y_tr)
    y_pred = xgb_fold.predict(X_te)
    y_prob = xgb_fold.predict_proba(X_te)[:, 1]
    m = evaluate_model(y_te, y_pred, y_prob)
    for k, v in m.items(): cv_results['XGBoost'][k].append(v)
    print(f'AUC={m["roc_auc"]:.4f}', end='  ')

    # MLP (clone pipeline with best params)
    print('MLP...', end=' ')
    mlp_fold = clone(mlp_fitted)
    mlp_fold.fit(X_tr, y_tr)
    y_pred = mlp_fold.predict(X_te)
    y_prob = mlp_fold.predict_proba(X_te)[:, 1]
    m = evaluate_model(y_te, y_pred, y_prob)
    for k, v in m.items(): cv_results['MLP'][k].append(v)
    print(f'AUC={m["roc_auc"]:.4f}', end='  ')

    # AE-MLP (same architecture, retrained each fold)
    print('AE-MLP...', end=' ')
    m, _, _ = train_eval_ae_mlp(X_tr, y_tr, X_te, y_te, input_dim, encoding_dim=16)
    for k, v in m.items(): cv_results['AE-MLP (d=16)'][k].append(v)
    print(f'AUC={m["roc_auc"]:.4f}')

print('\nCross-validation complete.')

EXPERIMENT 2: Five-Fold Stratified Cross-Validation
Using best hyperparameters from Experiment 1

RF params: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 200, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
XGB params: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 1.0, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': 0, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.2, 'max_bin': None, 'max_cat_threshold


# CELL 13: Print TABLE 9 - Cross-Validation Results

In [13]:
# ============================================================
# CELL 13: Print TABLE 9 - Cross-Validation Results
# ============================================================
print('=' * 70)
print('TABLE 9: Five-Fold Stratified Cross-Validation (Mean +/- Std)')
print('=' * 70)

print(f'{"Model":<18} {"Accuracy":>16} {"Precision":>16} {"Recall":>16} {"F1-Score":>16} {"ROC-AUC":>16}')
print('-' * 98)

cv_summary = {}
for name, metrics in cv_results.items():
    row = {}
    parts = []
    for mn in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        vals = np.array(metrics[mn])
        row[f'{mn}_mean'] = vals.mean()
        row[f'{mn}_std'] = vals.std()
        parts.append(f'{vals.mean():.4f}+/-{vals.std():.4f}')
    cv_summary[name] = row
    print(f'{name:<18} {parts[0]:>16} {parts[1]:>16} {parts[2]:>16} {parts[3]:>16} {parts[4]:>16}')

df_t9 = pd.DataFrame(cv_summary).T
df_t9.index.name = 'Model'
df_t9.to_csv('results/table9_cross_validation.csv')
print('\nSaved to results/table9_cross_validation.csv')

print('\n--- Raw Fold-Level ROC-AUC ---')
for name in cv_results:
    vals = cv_results[name]['roc_auc']
    print(f'  {name}: {[f"{v:.4f}" for v in vals]}')

TABLE 9: Five-Fold Stratified Cross-Validation (Mean +/- Std)
Model                      Accuracy        Precision           Recall         F1-Score          ROC-AUC
--------------------------------------------------------------------------------------------------
Random Forest       0.8614+/-0.0050  0.8680+/-0.0032  0.8525+/-0.0080  0.8602+/-0.0054  0.9435+/-0.0037
XGBoost             0.8949+/-0.0061  0.8938+/-0.0077  0.8964+/-0.0058  0.8951+/-0.0060  0.9680+/-0.0029
MLP                 0.8228+/-0.0058  0.8530+/-0.0147  0.7809+/-0.0252  0.8149+/-0.0090  0.9170+/-0.0027
AE-MLP (d=16)       0.7797+/-0.0056  0.8462+/-0.0139  0.6845+/-0.0270  0.7563+/-0.0115  0.8676+/-0.0061

Saved to results/table9_cross_validation.csv

--- Raw Fold-Level ROC-AUC ---
  Random Forest: ['0.9478', '0.9390', '0.9404', '0.9426', '0.9478']
  XGBoost: ['0.9705', '0.9645', '0.9646', '0.9694', '0.9710']
  MLP: ['0.9219', '0.9160', '0.9149', '0.9145', '0.9175']
  AE-MLP (d=16): ['0.8771', '0.8652', '0.8639', '0.85

# CELL 14: EXPERIMENT 3 - Wilcoxon Signed-Rank Tests

In [14]:
# ============================================================
# CELL 14: EXPERIMENT 3 - Wilcoxon Signed-Rank Tests
# ============================================================
print('=' * 70)
print('TABLE 10: Pairwise Wilcoxon Signed-Rank Tests (alpha = 0.05)')
print('=' * 70)

model_names = list(cv_results.keys())
pairs = []
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        pairs.append((model_names[i], model_names[j]))

print(f'{"Model Pair":<35} {"W(ROC-AUC)":>12} {"p(ROC-AUC)":>12} {"W(F1)":>12} {"p(F1)":>12}')
print('-' * 83)

wilcoxon_data = []
for m1, m2 in pairs:
    roc1 = np.array(cv_results[m1]['roc_auc'])
    roc2 = np.array(cv_results[m2]['roc_auc'])
    f1_1 = np.array(cv_results[m1]['f1'])
    f1_2 = np.array(cv_results[m2]['f1'])

    try:
        w_roc, p_roc = wilcoxon(roc1, roc2, alternative='two-sided')
    except ValueError:
        w_roc, p_roc = float('nan'), 1.0

    try:
        w_f1, p_f1 = wilcoxon(f1_1, f1_2, alternative='two-sided')
    except ValueError:
        w_f1, p_f1 = float('nan'), 1.0

    pair_name = f'{m1} vs {m2}'
    print(f'{pair_name:<35} {w_roc:>12.1f} {p_roc:>12.4f} {w_f1:>12.1f} {p_f1:>12.4f}')
    wilcoxon_data.append({
        'Model Pair': pair_name,
        'W (ROC-AUC)': w_roc, 'p (ROC-AUC)': p_roc,
        'W (F1-Score)': w_f1, 'p (F1-Score)': p_f1
    })

# Sign test
print('\n--- Sign Test Interpretation ---')
for m1, m2 in pairs:
    roc1 = np.array(cv_results[m1]['roc_auc'])
    roc2 = np.array(cv_results[m2]['roc_auc'])
    wins = (roc1 > roc2).sum()
    losses = (roc1 < roc2).sum()
    ties = (roc1 == roc2).sum()
    sign_p = 0.5 ** max(wins, losses) if wins + losses > 0 else 1.0
    winner = m1 if wins > losses else m2
    print(f'  {m1} vs {m2}: {wins}W-{losses}L-{ties}T '
          f'(one-sided p = {sign_p:.5f}, favour: {winner})')

df_t10 = pd.DataFrame(wilcoxon_data)
df_t10.to_csv('results/table10_wilcoxon_tests.csv', index=False)
print('\nSaved to results/table10_wilcoxon_tests.csv')

TABLE 10: Pairwise Wilcoxon Signed-Rank Tests (alpha = 0.05)
Model Pair                            W(ROC-AUC)   p(ROC-AUC)        W(F1)        p(F1)
-----------------------------------------------------------------------------------
Random Forest vs XGBoost                     0.0       0.0625          0.0       0.0625
Random Forest vs MLP                         0.0       0.0625          0.0       0.0625
Random Forest vs AE-MLP (d=16)               0.0       0.0625          0.0       0.0625
XGBoost vs MLP                               0.0       0.0625          0.0       0.0625
XGBoost vs AE-MLP (d=16)                     0.0       0.0625          0.0       0.0625
MLP vs AE-MLP (d=16)                         0.0       0.0625          0.0       0.0625

--- Sign Test Interpretation ---
  Random Forest vs XGBoost: 0W-5L-0T (one-sided p = 0.03125, favour: XGBoost)
  Random Forest vs MLP: 5W-0L-0T (one-sided p = 0.03125, favour: Random Forest)
  Random Forest vs AE-MLP (d=16): 5W-0L-0T (one

# CELL 15: EXPERIMENT 4 - AE-MLP Ablation Study

In [ ]:
# ============================================================
# CELL 15: EXPERIMENT 4 - AE-MLP Ablation Study
# ============================================================
print('=' * 70)
print('EXPERIMENT 4: AE-MLP Ablation Study (Bottleneck Dimensionality)')
print('=' * 70)

bottleneck_dims = [8, 16, 24, 32]
input_dim = X.shape[1]
skf_ab = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

ablation_results = {}

for d in bottleneck_dims:
    ratio = f'1:{input_dim/d:.1f}'
    print(f'\n--- Bottleneck d={d} (compression ratio {ratio}) ---')
    fold_m = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []}

    for fold, (train_idx, test_idx) in enumerate(skf_ab.split(X, y), 1):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        print(f'  Fold {fold}...', end=' ')
        m, _, _ = train_eval_ae_mlp(X_tr, y_tr, X_te, y_te, input_dim, encoding_dim=d)
        for k, v in m.items(): fold_m[k].append(v)
        print(f'Recall={m["recall"]:.4f} AUC={m["roc_auc"]:.4f}')

    ablation_results[d] = {'ratio': ratio, 'metrics': fold_m}

print('\nAblation study complete.')

EXPERIMENT 4: AE-MLP Ablation Study (Bottleneck Dimensionality)

--- Bottleneck d=8 (compression ratio 1:4.6) ---
  Fold 1... Recall=0.6660 AUC=0.8512
  Fold 2... Recall=0.6268 AUC=0.8374
  Fold 3... Recall=0.6382 AUC=0.8342
  Fold 4... Recall=0.6740 AUC=0.8291
  Fold 5... Recall=0.6720 AUC=0.8388

--- Bottleneck d=16 (compression ratio 1:2.3) ---
  Fold 1... Recall=0.6921 AUC=0.8824
  Fold 2... Recall=0.7313 AUC=0.8704
  Fold 3... Recall=0.6700 AUC=0.8637
  Fold 4... Recall=0.7078 AUC=0.8723
  Fold 5... Recall=0.6804 AUC=0.8665

--- Bottleneck d=24 (compression ratio 1:1.5) ---
  Fold 1... Recall=0.6767 AUC=0.8830
  Fold 2... Recall=0.7075 AUC=0.8830
  Fold 3... Recall=0.7406 AUC=0.8771
  Fold 4... Recall=0.6690 AUC=0.8833
  Fold 5... Recall=0.7149 AUC=0.8774

--- Bottleneck d=32 (compression ratio 1:1.2) ---
  Fold 1... Recall=0.6774 AUC=0.8871
  Fold 2... Recall=0.7554 AUC=0.8848
  Fold 3... Recall=0.6898 AUC=0.8754
  Fold 4... Recall=0.6928 AUC=0.8860
  Fold 5... Recall=0.7610 AUC=

# CELL 16: Print TABLE 11 - Ablation Results

In [ ]:
# ============================================================
# CELL 16: Print TABLE 11 - Ablation Results
# ============================================================
print('=' * 70)
print('TABLE 11: AE-MLP Ablation Study - Performance by Bottleneck Size')
print('=' * 70)

print(f'{"d":>4} {"Ratio":>8} {"Accuracy":>18} {"Precision":>18} {"Recall":>18} {"F1-Score":>18} {"ROC-AUC":>18}')
print('-' * 104)

ablation_data = []
for d in bottleneck_dims:
    r = ablation_results[d]
    m = r['metrics']
    row = {'Bottleneck (d)': d, 'Compression Ratio': r['ratio']}
    parts = []
    for mn in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        vals = np.array(m[mn])
        row[f'{mn}_mean'] = vals.mean()
        row[f'{mn}_std'] = vals.std()
        parts.append(f'{vals.mean():.4f}+/-{vals.std():.4f}')
    print(f'{d:>4} {r["ratio"]:>8} {parts[0]:>18} {parts[1]:>18} {parts[2]:>18} {parts[3]:>18} {parts[4]:>18}')
    ablation_data.append(row)

# Key calculations
rec_8 = np.array(ablation_results[8]['metrics']['recall']).mean()
rec_24 = np.array(ablation_results[24]['metrics']['recall']).mean()
rec_32 = np.array(ablation_results[32]['metrics']['recall']).mean()
rf_recall_cv = np.array(cv_results['Random Forest']['recall']).mean()

print(f'\n--- Key Findings ---')
print(f'  Recall improvement d=8 to d=24: {((rec_24 - rec_8) / rec_8) * 100:.1f}%')
print(f'  Best AE-MLP config: d=24 (recall {rec_24:.4f})')
print(f'  Best AE-MLP recall ({rec_24:.4f}) vs RF CV recall ({rf_recall_cv:.4f}): '
      f'{((rf_recall_cv - rec_24) / rf_recall_cv) * 100:.1f}% deficit')

df_t11 = pd.DataFrame(ablation_data)
df_t11.to_csv('results/table11_ablation_study.csv', index=False)
print('\nSaved to results/table11_ablation_study.csv')

TABLE 11: AE-MLP Ablation Study - Performance by Bottleneck Size
   d    Ratio           Accuracy          Precision             Recall           F1-Score            ROC-AUC
--------------------------------------------------------------------------------------------------------
   8    1:4.6    0.7537+/-0.0073    0.8168+/-0.0203    0.6554+/-0.0192    0.7268+/-0.0085    0.8381+/-0.0073
  16    1:2.3    0.7822+/-0.0079    0.8416+/-0.0201    0.6963+/-0.0215    0.7616+/-0.0098    0.8711+/-0.0064
  24    1:1.5    0.7913+/-0.0018    0.8566+/-0.0257    0.7017+/-0.0261    0.7706+/-0.0055    0.8808+/-0.0029
  32    1:1.2    0.7966+/-0.0063    0.8564+/-0.0284    0.7153+/-0.0355    0.7782+/-0.0114    0.8867+/-0.0078

--- Key Findings ---
  Recall improvement d=8 to d=24: 7.1%
  Best AE-MLP config: d=24 (recall 0.7017)
  Best AE-MLP recall (0.7017) vs RF CV recall (0.8525): 17.7% deficit

Saved to results/table11_ablation_study.csv


# CELL 17: EXPERIMENT 5 - Permutation Feature Importance


In [ ]:
# ============================================================
# CELL 17: EXPERIMENT 5 - Permutation Feature Importance
# ============================================================
print('=' * 70)
print('EXPERIMENT 5: Permutation Feature Importance (10 repeats)')
print('=' * 70)

print('Computing RF importance...')
rf_perm = permutation_importance(
    rf_fitted, X_test, y_test,
    n_repeats=10, random_state=SEED, scoring='roc_auc', n_jobs=-1
)

print('Computing XGBoost importance...')
xgb_perm = permutation_importance(
    xgb_fitted, X_test, y_test,
    n_repeats=10, random_state=SEED, scoring='roc_auc', n_jobs=-1
)

# Build ranking
df_imp = pd.DataFrame({
    'Feature': feature_names,
    'RF': rf_perm.importances_mean,
    'XGBoost': xgb_perm.importances_mean,
})
df_imp['Ensemble Mean'] = (df_imp['RF'] + df_imp['XGBoost']) / 2
df_imp = df_imp.sort_values('Ensemble Mean', ascending=False).reset_index(drop=True)
df_imp.index = df_imp.index + 1
df_imp.index.name = 'Rank'

print('\n' + '=' * 70)
print('TABLE 8: Feature Importance Ranking (Top 10)')
print('=' * 70)

top10 = df_imp.head(10)
print(f'{"Rank":>4} {"Feature":<30} {"RF":>10} {"XGBoost":>10} {"Ens.Mean":>10}')
print('-' * 68)
for rank, row in top10.iterrows():
    print(f'{rank:>4} {row["Feature"]:<30} {row["RF"]:>10.4f} {row["XGBoost"]:>10.4f} {row["Ensemble Mean"]:>10.4f}')

# Spearman correlation
rf_ranks = df_imp['RF'].rank(ascending=False)
xgb_ranks = df_imp['XGBoost'].rank(ascending=False)
rho, p_val = spearmanr(rf_ranks.head(10), xgb_ranks.head(10))
print(f'\nSpearman rank correlation (top 10): rho = {rho:.4f}, p = {p_val:.6f}')

# Check temporal features
temporal = ['first_entry', 'last_exit', 'first_entry_time', 'last_exit_time',
            'entry_time', 'exit_time', 'login_time', 'first_entry_minutes',
            'last_exit_minutes']
temporal_found = [f for f in temporal if f in top10['Feature'].values]
if temporal_found:
    print(f'Temporal features in top 10: {temporal_found}')
else:
    print('No temporal access features in top 10 (confirms SHAP findings)')

df_imp.to_csv('results/table8_feature_importance.csv')
print('\nSaved to results/table8_feature_importance.csv')

EXPERIMENT 5: Permutation Feature Importance (10 repeats)
Computing RF importance...
Computing XGBoost importance...

TABLE 8: Feature Importance Ranking (Top 10)
Rank Feature                                RF    XGBoost   Ens.Mean
--------------------------------------------------------------------
   1 employee_seniority_years           0.1211     0.1614     0.1412
   2 employee_classification            0.0806     0.0885     0.0846
   3 total_files_burned                 0.0352     0.0497     0.0425
   4 has_foreign_citizenship            0.0385     0.0317     0.0351
   5 is_contractor                      0.0264     0.0252     0.0258
   6 num_unique_campus                  0.0265     0.0192     0.0228
   7 has_medical_history                0.0232     0.0191     0.0212
   8 num_bw_prints                      0.0167     0.0130     0.0148
   9 has_criminal_record                0.0118     0.0169     0.0143
  10 num_print_commands                 0.0056     0.0178     0.0117

Spearman

# CELL 18: Summary + Download all results


In [ ]:
# ============================================================
# CELL 18: Summary + Download all results
# ============================================================
print('=' * 70)
print('PIPELINE COMPLETE - ALL TABLES GENERATED')
print('=' * 70)

print('\nOutput files:')
for f in sorted(os.listdir('results')):
    if f.endswith('.csv'):
        size = os.path.getsize(f'results/{f}')
        print(f'  results/{f} ({size:,} bytes)')

import shutil
shutil.make_archive('thesis_results', 'zip', 'results')
print(f'\nAll results zipped to: thesis_results.zip')

from google.colab import files
files.download('thesis_results.zip')
print('Download started.')

PIPELINE COMPLETE - ALL TABLES GENERATED

Output files:
  results/table10_wilcoxon_tests.csv (331 bytes)
  results/table11_ablation_study.csv (982 bytes)
  results/table6_test_performance.csv (454 bytes)
  results/table7_confusion_matrices.csv (150 bytes)
  results/table8_feature_importance.csv (3,193 bytes)
  results/table9_cross_validation.csv (961 bytes)

All results zipped to: thesis_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.
